# 📋 Bioinformatics Quick Reference Card

Essential code patterns for bioinformatics in Python

---

## 🧬 DNA/RNA Operations

### Basic Sequence Handling

In [ ]:
# Nucleotide counting
seq = "ATGCGATCGATCG"
counts = {base: seq.count(base) for base in 'ATGC'}

# GC content
gc_content = (seq.count('G') + seq.count('C')) / len(seq) * 100

# Complement
complement = seq.translate(str.maketrans('ATGC', 'TACG'))

# Reverse complement
rev_comp = seq.translate(str.maketrans('ATGC', 'TACG'))[::-1]

# DNA to RNA transcription
rna = seq.replace('T', 'U')

# Extract codons
codons = [seq[i:i+3] for i in range(0, len(seq)-2, 3)]

print(f"Sequence: {seq}")
print(f"Counts: {counts}")
print(f"GC: {gc_content:.1f}%")
print(f"RevComp: {rev_comp}")
print(f"Codons: {codons}")

### Codon Table & Translation

In [ ]:
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate(dna: str) -> str:
    """Translate DNA to protein."""
    protein = []
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3].upper()
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)

print(translate("ATGGCTTAA"))  # MA

---

## 📁 File Parsing

### FASTA Parser

In [ ]:
def parse_fasta(filename: str) -> dict:
    """Parse FASTA file to dictionary."""
    sequences = {}
    current_id = None
    current_seq = []
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id:
                    sequences[current_id] = ''.join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            elif line:
                current_seq.append(line)
        
        if current_id:
            sequences[current_id] = ''.join(current_seq)
    
    return sequences

# Usage: seqs = parse_fasta('sequences.fasta')

### FASTA Writer

In [ ]:
def write_fasta(sequences: dict, filename: str, width: int = 80):
    """Write sequences to FASTA file."""
    with open(filename, 'w') as f:
        for seq_id, seq in sequences.items():
            f.write(f">{seq_id}\n")
            for i in range(0, len(seq), width):
                f.write(seq[i:i+width] + "\n")

# Usage: write_fasta({'seq1': 'ATGC...', 'seq2': 'GGCC...'}, 'output.fasta')

### FASTA Generator (Memory Efficient)

In [ ]:
def fasta_reader(filename: str):
    """Generator for reading large FASTA files."""
    with open(filename, 'r') as f:
        header, seq = None, []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    yield header, ''.join(seq)
                header = line[1:]
                seq = []
            elif line:
                seq.append(line)
        if header:
            yield header, ''.join(seq)

# Usage:
# for header, sequence in fasta_reader('large.fasta'):
#     process(sequence)

---

## 🔬 BioPython Essentials

In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Create sequence
dna = Seq("ATGCGATCGATCG")

# Operations
print(f"Sequence: {dna}")
print(f"Complement: {dna.complement()}")
print(f"Rev Comp: {dna.reverse_complement()}")
print(f"Transcribe: {dna.transcribe()}")
print(f"Translate: {dna.translate()}")

In [ ]:
# Parse FASTA with BioPython
# for record in SeqIO.parse('file.fasta', 'fasta'):
#     print(f"{record.id}: {len(record.seq)} bp")

# Parse GenBank
# for record in SeqIO.parse('file.gb', 'genbank'):
#     for feature in record.features:
#         if feature.type == 'CDS':
#             print(feature.qualifiers.get('gene', ['unknown']))

# Write sequences
# SeqIO.write(records, 'output.fasta', 'fasta')

### NCBI Entrez Access

In [ ]:
from Bio import Entrez

# ALWAYS set your email!
Entrez.email = "your.email@example.com"

# Search for sequences
# handle = Entrez.esearch(db="nucleotide", term="human[orgn] AND insulin", retmax=5)
# record = Entrez.read(handle)
# ids = record["IdList"]

# Fetch sequence by ID
# handle = Entrez.efetch(db="nucleotide", id="NM_000207", rettype="fasta", retmode="text")
# seq_record = SeqIO.read(handle, "fasta")

---

## 📊 Data Analysis Patterns

### GC Content Sliding Window

In [ ]:
import numpy as np

def gc_sliding_window(seq: str, window: int = 100, step: int = 10):
    """Calculate GC content in sliding windows."""
    positions = []
    gc_values = []
    
    for i in range(0, len(seq) - window + 1, step):
        window_seq = seq[i:i+window].upper()
        gc = (window_seq.count('G') + window_seq.count('C')) / window * 100
        positions.append(i + window // 2)
        gc_values.append(gc)
    
    return np.array(positions), np.array(gc_values)

# Usage:
# pos, gc = gc_sliding_window(sequence, window=500, step=100)
# plt.plot(pos, gc)

### K-mer Counting

In [ ]:
from collections import Counter

def count_kmers(seq: str, k: int = 3) -> dict:
    """Count k-mers in sequence."""
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    return Counter(kmers)

# All possible k-mers
from itertools import product
def all_kmers(k: int, alphabet: str = 'ATGC'):
    return [''.join(p) for p in product(alphabet, repeat=k)]

print(count_kmers("ATGATGATG", k=3))

### ORF Finding

In [ ]:
def find_orfs(seq: str, min_length: int = 30):
    """Find all ORFs in a sequence."""
    import re
    orfs = []
    seq = seq.upper()
    
    # Search all 3 frames
    for frame in range(3):
        # Find all start codons
        for match in re.finditer('ATG', seq[frame:]):
            start = match.start() + frame
            
            # Find stop codon in same frame
            for i in range(start + 3, len(seq) - 2, 3):
                codon = seq[i:i+3]
                if codon in ('TAA', 'TAG', 'TGA'):
                    length = i + 3 - start
                    if length >= min_length:
                        orfs.append({
                            'start': start,
                            'end': i + 3,
                            'frame': frame,
                            'length': length,
                            'sequence': seq[start:i+3]
                        })
                    break
    
    return sorted(orfs, key=lambda x: x['length'], reverse=True)

---

## 🎨 Visualization Templates

### GC Content Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_gc_content(positions, gc_values, title="GC Content"):
    """Plot GC content along sequence."""
    fig, ax = plt.subplots(figsize=(12, 4))
    
    ax.plot(positions, gc_values, 'b-', linewidth=0.5)
    ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% GC')
    ax.fill_between(positions, gc_values, alpha=0.3)
    
    ax.set_xlabel('Position (bp)')
    ax.set_ylabel('GC Content (%)')
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    return fig, ax

### Sequence Logo (Simplified)

In [ ]:
def plot_nucleotide_frequencies(sequences: list, title="Nucleotide Frequency"):
    """Plot nucleotide frequencies at each position."""
    import numpy as np
    import matplotlib.pyplot as plt
    
    length = len(sequences[0])
    freqs = {base: np.zeros(length) for base in 'ATGC'}
    
    for seq in sequences:
        for i, base in enumerate(seq.upper()):
            if base in freqs:
                freqs[base][i] += 1
    
    for base in freqs:
        freqs[base] /= len(sequences)
    
    fig, ax = plt.subplots(figsize=(max(8, length * 0.5), 4))
    positions = np.arange(length)
    
    bottom = np.zeros(length)
    colors = {'A': 'green', 'T': 'red', 'G': 'orange', 'C': 'blue'}
    
    for base in 'ATGC':
        ax.bar(positions, freqs[base], bottom=bottom, label=base, color=colors[base])
        bottom += freqs[base]
    
    ax.set_xlabel('Position')
    ax.set_ylabel('Frequency')
    ax.set_title(title)
    ax.legend()
    ax.set_xticks(positions)
    
    plt.tight_layout()
    return fig, ax

### Heatmap for Gene Expression

In [ ]:
import seaborn as sns

def plot_expression_heatmap(data, genes=None, samples=None):
    """Plot gene expression heatmap with clustering."""
    fig = plt.figure(figsize=(10, 8))
    
    g = sns.clustermap(
        data,
        cmap='RdBu_r',
        center=0,
        xticklabels=samples if samples else True,
        yticklabels=genes if genes else True,
        figsize=(10, 8)
    )
    
    return g

---

## 🔧 Utility Functions

### Sequence Validation

In [ ]:
def validate_dna(seq: str) -> bool:
    """Check if sequence is valid DNA."""
    return set(seq.upper()) <= set('ATGCN')

def validate_protein(seq: str) -> bool:
    """Check if sequence is valid protein."""
    valid = set('ACDEFGHIKLMNPQRSTVWY*')
    return set(seq.upper()) <= valid

def detect_sequence_type(seq: str) -> str:
    """Detect if sequence is DNA, RNA, or protein."""
    seq = seq.upper()
    
    if 'U' in seq and 'T' not in seq:
        return 'RNA'
    elif set(seq) <= set('ATGCN'):
        return 'DNA'
    else:
        return 'Protein'

### Molecular Weight Calculation

In [ ]:
def dna_molecular_weight(seq: str) -> float:
    """Calculate DNA molecular weight (Da)."""
    weights = {'A': 331.2, 'T': 322.2, 'G': 347.2, 'C': 307.2}
    return sum(weights.get(base, 0) for base in seq.upper())

AMINO_ACID_WEIGHTS = {
    'A': 89.1, 'R': 174.2, 'N': 132.1, 'D': 133.1, 'C': 121.2,
    'E': 147.1, 'Q': 146.2, 'G': 75.1, 'H': 155.2, 'I': 131.2,
    'L': 131.2, 'K': 146.2, 'M': 149.2, 'F': 165.2, 'P': 115.1,
    'S': 105.1, 'T': 119.1, 'W': 204.2, 'Y': 181.2, 'V': 117.1
}

def protein_molecular_weight(seq: str) -> float:
    """Calculate protein molecular weight (Da)."""
    # Subtract water for peptide bonds
    weight = sum(AMINO_ACID_WEIGHTS.get(aa, 0) for aa in seq.upper())
    water = 18.0 * (len(seq) - 1) if len(seq) > 0 else 0
    return weight - water

---

## 📐 Regular Expression Patterns

Common bioinformatics regex patterns:

In [ ]:
import re

# Common patterns
PATTERNS = {
    'start_codon': r'ATG',
    'stop_codons': r'TAA|TAG|TGA',
    'kozak': r'[AG]CC.{0,3}ATG[G]',  # Kozak consensus
    'tata_box': r'TATA[AT]A[AT]',
    'poly_a': r'AATAAA',
    'restriction_ecori': r'GAATTC',
    'restriction_bamhi': r'GGATCC',
    'restriction_hindiii': r'AAGCTT',
    'n_glycosylation': r'N[^P][ST][^P]',  # Protein motif
    'nuclear_localization': r'[KR]{4,}',  # Basic NLS
}

def find_motif(sequence: str, pattern: str) -> list:
    """Find all occurrences of a motif."""
    return [(m.start(), m.end(), m.group()) 
            for m in re.finditer(pattern, sequence.upper())]

# Example
seq = "ATGCGAATTCGATGCAAGCTTGGATCC"
for name, pattern in [('EcoRI', PATTERNS['restriction_ecori']),
                       ('BamHI', PATTERNS['restriction_bamhi'])]:
    matches = find_motif(seq, pattern)
    print(f"{name}: {matches}")

---

## 💾 One-Hot Encoding for ML

In [ ]:
import numpy as np

def one_hot_encode(seq: str) -> np.ndarray:
    """One-hot encode a DNA sequence."""
    mapping = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    encoded = np.zeros((4, len(seq)), dtype=np.float32)
    
    for i, base in enumerate(seq.upper()):
        if base in mapping:
            encoded[mapping[base], i] = 1.0
    
    return encoded

# Example
encoded = one_hot_encode("ATGC")
print("One-hot encoding of ATGC:")
print(encoded)

---

## 📎 Import Cheatsheet

```python
# Core Python
import os
import sys
import re
from collections import Counter, defaultdict
from itertools import product, combinations
from functools import lru_cache

# Data Science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# BioPython
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO, Entrez, AlignIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.PDB import PDBParser

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
```

---

**Keep this notebook open as a quick reference while working!** 🧬

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/01_QUICK_REFERENCE.ipynb`)
